<h2>Impact (v0.64) - 'Impact (v0.64).ipynb'</h2>

<u><h4>Context</h4></u>

The aim of the 'Impact' notebooks are to explore economic impacts resulsting from the FEM outputs in the lead-up to Autumn 2026, taking foward outputs from the associated 'Trade' notebooks.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios. Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.6' was amended to work with Comtrade import and export trade portfolios downloaded from the platform. 'v0.61' made use of the updated 'trade' notebooks that derive a new weight field using Value-to-Weight and Weight-to-Value ratios observed in the CEPII BACI datasets across 2017-23. In the case of Trinidad and Tobaogo, import volumes were around 65% higher under the WtV approach which corresponded to a 39% greater 2041 BAU impact from FEM costs, but this was slightly less for exports where the weighted WtV was 9% more but 2041 FEM costs under BAU were around 10% less. A new 'composite' weight field has been added in Trade (v0.61) which only introduces the VtW-derived values where 'netWgt' is observed to be zero, however this resulted in only a tiny increase in recorded weight of around 1% for Angola, whereas the full VtW-derived field was ~50% greater. 

'v0.62' makes use of a new 'transport cost' field derived using ad-valorem transport cost rates identified from UNCTAD's Trade-and-Transport database. With these, we can now introduce another results disaggregation that presents NZF costs relative to expected transport costs, which is then fully in-line with how impacts are expressed in UNCTAD's CIA Task 3 work. 'v0.63' includes an assessment of 'Cost Intensity' matching process (similar to CIF-FOB matching), plus metrics on 'fac_vessel_types' and 'average sea-distance' (introduced in Trade v0.63). 'v0.64' has been modified throughout to account for a new scenario, namely the 'Japan' proposal, plus to remove the 'Low-then-High' scenario. In terms of the Impact notebook, this remains the same as 'v0.63'.

In [1]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

version = "v0.64"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
voys_fem_dir = "/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments (v0.2)/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_v0.2.csv"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"
res_mlc = True

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "662"
iso_country = "Saint Lucia"
year = 2020
gdp = 1.511e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."
os.makedirs(output_dir+"{0}/results/".format(iso_country)) if not os.path.exists(output_dir+"{0}/results/".format(iso_country)) else "Output results folder already exists."

## 1.2 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv(voys_fem_dir, dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,0,5034525,413781699,CNTAX,156,CHN,CN,China,Taixing,CNNKG,156,CHN,CN,China,Nanjing,2018-01-01 06:00:00,2018-01-01 19:00:00,13.0,1.0,14.0,62.568523,0.0,0.0,0.0,3.635661,0.338483,3.974144,0.0,0.0,0.0,0.0,0.0,0.0,0.155243,0.014453,11.655930,1.085176,12.741107,142,Asia,030,Eastern Asia,upp_mid_income,0,0,142,Asia,030,Eastern Asia,upp_mid_income,0,0,0.169696,Refrigerated bulk,3786.0,dwt,3881.0,NaN,NaN,NaN,NaN,2,2000-5999,1959.0,Panama,SSD,4,76.1,0.666667,0.03155,0.032676,0.032143,0.031676,0.031218,0.030769,0.03031,0.029878,0.029438,0.031576,0.052178,0.052414,0.05804,0.060001,0.059297,0.058435,0.058228,0.058319,0.031568,0.034644,0.036843,0.044726,0.049621,0.053882,0.057797,0.059065,0.057999,0.031568,0.032727,0.034529,0.040735,0.048178,0.055864,0.057447,0.058672,0.05942,0.031559,0.032699,0.033808,0.037977,0.040129,0.041837,0.043315,0.044697,0.045876,0.031559,0.033985,0.034896,0.036131,0.038297,0.039924,0.040606,0.041099,0.041463,242828.439338,7661.320668,7934.549945,7805.304846,7691.950531,7580.696290,7471.542086,7360.199865,7255.307537,7148.350725,7667.458038,12670.391275,12727.642802,14093.647297,14570.027623,14398.931835,14189.590704,14139.488873,14161.462401,7665.715352,8412.516422,8946.6

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [4]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str", "HS4":"str"}
if res_mlc == False:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_{1}.csv".format(iso_country, version), dtype=trade_im_d)
else:
    trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_mlc_{1}.csv".format(iso_country, version), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Saint Lucia': 

Number of trade records present in dataset: 15911
Number of partner countries present in dataset: 124
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$437.6m
Total weight within the dataset: 447.4 million kilograms

Distance to market: 82.6 nm (min) to 13462.6 nm (max), 4516.6 nm (observed mean)
Average distance to market (volume-weighted): 1790.3 nm
Average distance to market (value-weighted): 3275.1 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue
0,0,662,Saint Lucia,028,Antigua and Barbuda,000,World,Import,Sea,040299 | Dairy produce; milk,40299,kg,1.118286,1.118286,7.268859,0.0,7.268859,Antigua and Barbuda | 040299 | Dairy produce; ...,04,04 | Dairy produce; birds,Antigua and Barbuda | 04 | Dairy produce; birds,0402,0402 | Milk and cream; conc,Antigua and Barbuda | 0402 | Milk and cream; conc,4.0,402.0,40299.0,"Milk & cream, concentrated (excl. in powder), ...",Agriculture,1520.0,Manufacture of dairy products,2224.0,"Milk & cream, concentrated, not in solid form,...",22.0,MIL - Dairy products,2292.0,"Milk and cream, concentrated or containing add...",2.8314,0.3532,both,0.153846,2.567361,1.118286,[Container],['Container'],392.0,243.577432,0.625351,3 - Reporter and HS4 code,5.38,19,0.37,7.27,6.90
1,1,662,Saint Lucia,028,Antigua and Barbuda,000,World,Import,Sea,"081340 | Fruit, edible; fruit",81340,kg,6.807204,6.807204,16.850781,0.0,16.850781,"Antigua and Barbuda | 081340 | Fruit, edible; ...",08,"08 | Fruit and nuts, edib","Antigua and Barbuda | 08 | Fruit and nuts, edib",0813,"0813 | Fruit, dried, other","Antigua and Barbuda | 0813 | Fruit, dried, other",8.0,813.0,81340.0,Dried fruit (excl. of 08.01-08.06 & 0813.10-08...,Agriculture,113.0,"Growing of fruit, nuts, beverage and spice crops",5799.0,"Fruit, dried, n.e.s., & mixtures, n.e.s., of n...",4.0,"V_F - Vegetables, fruit, nuts",135.0,"Other fruit, dried",2.9373,0.3405,both,0.403970,5.737691,6.807204,[Container],['Container'],392.0,243.577432,1.397572,3 - Reporter and HS4 code,4.79,6,0.77,16.85,16.08


## 2.2 Derive Internationally Arriving Voyages

In [5]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 340
Total size of the fleet: 135 vessels
6 vessel types contained in the dataset: ['Vehicle' 'Oil tanker' 'General cargo' 'Container' 'Chemical tanker'
 'Bulk carrier']
Total transport work capacity: 13053.7 million deadweight-tonne nautical miles
Total cargo transported: 5887395.2 tonnes
Total transport work: 6512.4 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 1117.0 EJ
Total CO2 produced: 86238.3 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,79197,9078830,353910000,MQFDF,474,MTQ,MQ,Martinique,Fort De France,LCCAS,662,LCA,LC,Saint Lucia,Castries,2018-12-07 14:00:00,2018-12-07 23:00:00,9.0,4.0,13.0,57.117283,4.077641,1.118897,5.196538,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.163921,0.044980,12.697774,3.484246,16.182020,019,Americas,419,Latin America and the Caribbean,high_income,0,1,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,1,0.208901,Vehicle,49012.0,gt,15553.0,NaN,NaN,NaN,5008.0,2,30000-49999,1994.0,Panama,SSD,1,21.8,0.292985,0.010079,0.010502,0.010302,0.010127,0.009955,0.009786,0.009614,0.009452,0.009287,0.010033,0.01817,0.016375,0.018434,0.020218,0.018328,0.017672,0.015414,0.014738,0.01004,0.011031,0.011371,0.01404,0.016494,0.016992,0.014587,0.017643,0.015442,0.01004,0.010325,0.010829,0.012868,0.015565,0.018792,0.016586,0.019151,0.01784,0.01006,0.010413,0.010741,0.012181,0.012203,0.013616,0.011002,0.014594,0.012311,0.01006,0.01089,0.011148,0.011518,0.012217,0.012709,0.012855,0.01293,0.012958,8.883451e+05,8954.032668,9329.216910,9151.744257,8996.092004,8843.323462,8693.438581,8540.549229,8396.516536,8249.649000,8912.807106,16140.912265,14546.410337,16375.314155,17960.949405,16281.819768,15698.911579,1

## 2.3 Evaluate NZF Costs

In [6]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_arr_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_arr_voys_vess = int_arr_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_arr_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

                if int_arr_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_arr_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the importer.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "direct35_usd_dwtnm_"+year))
    
    if cost_int_match_desc_bool == False:
        trade_im["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True
    
    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd
    trade_im["direct_usd_"+year] = direct_usd

print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print("\nNumber of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_im.sea_dist_nm.min(), 1),
    round(trade_im.sea_dist_nm.max(), 1),
    round(trade_im.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_im.vol_kg_vtw / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_im.cifvalue / 1000 * trade_im.sea_dist_nm).sum() / (trade_im.cifvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.direct_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_agg = trade_im.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg["vol_kg_vtw_pct"] = round(100 * trade_im_agg.vol_kg_vtw / trade_im_agg.vol_kg_vtw.sum(), 1)
trade_im_agg["cifvalue_pct"] = round(100 * trade_im_agg.cifvalue / trade_im_agg.cifvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_im_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5)
))
trade_im_agg_2 = trade_im.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "cifvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_im_agg_2["vol_kg_vtw_pct"] = round(100 * trade_im_agg_2.vol_kg_vtw / trade_im_agg_2.vol_kg_vtw.sum(), 1)
trade_im_agg_2["cifvalue_pct"] = round(100 * trade_im_agg_2.cifvalue / trade_im_agg_2.cifvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_im_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "cifvalue_pct"]].head(5))

trade_im_na = trade_im[(trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
trade_im = trade_im[(~trade_im[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Liquefied gas tanker']

Global voyages table used for the following vessel types found: 

['Other liquids tankers' 'Cruise' 'Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['2106 | Food preparations no' '4911 | Printed matter, n.e.'
 '8517 | Telephone sets, incl' '2009 | Fruit or nut juices '
 '0802 | Nuts (excluding coco' '1806 | Chocolate and other '
 '3402 | Organic surface-acti' '4420 | Wood marquetry and i'
 '1905 | Bread, pastry, cakes' '1704 | Sugar confectionery '
 '3926 | Articles of plastics' '9401 | Seats (not those of '
 '9405 | Luminaires and light' '1008 | Buckwheat, millet an'
 '3822 | Reagents; diagnostic' 

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,cifvalue_pct
0,['Container'],1. Primary Int. Inv. with Vess Type,52.6,60.4
1,['Chemical tanker'],1. Primary Int. Inv. with Vess Type,11.1,6.5
2,['Bulk carrier'],1. Primary Int. Inv. with Vess Type,10.8,1.4
3,['Oil tanker'],1. Primary Int. Inv. with Vess Type,9.3,6.2
4,['all'],3. Primary Int. Inv. without Vess Type,6.7,12.0


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_im_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,662,Saint Lucia,028,Antigua and Barbuda,000,World,Import,Sea,040299 | Dairy produce; milk,40299,kg,1.118286,1.118286,7.268859,0.0,7.268859,Antigua and Barbuda | 040299 | Dairy produce; ...,04,04 | Dairy produce; birds,Antigua and Barbuda | 04 | Dairy produce; birds,0402,0402 | Milk and cream; conc,Antigua and Barbuda | 0402 | Milk and cream; conc,4.0,402.0,40299.0,"Milk & cream, concentrated (excl. in powder), ...",Agriculture,1520.0,Manufacture of dairy products,2224.0,"Milk & cream, concentrated, not in solid form,...",22.0,MIL - Dairy products,2292.0,"Milk and cream, concentrated or containing add...",2.8314,0.3532,both,0.153846,2.567361,1.118286,[Container],['Container'],392.0,243.577432,0.625351,3 - Reporter and HS4 code,5.38,19,0.37,7.27,6.90,1. Primary Int. Inv. with Vess Type,0.003269,0.003263,0.003263,0.003263,0.003266,0.003266,0.003419,0.006060,0.003771,0.003501,0.003511,0.003599,0.003348,0.005935,0.003970,0.003636,0.003553,0.003714,0.003285,0.006453,0.004923,0.004305,0.003983,0.003871,0.003224,0.006862,0.005646,0.005291,0.004264,0.004148,0.003164,0.006953,0.006643,0.006700,0.004623,0.004358,0.003103,0.00681,0.006526,0.007151,0.005289,0.004438,0.003045,0.006075,0.006332,0.006635,0.004974,0.004490,0.002986,0.005580,0.005673,0.007076,0.005846,0.004520
1,1,662,Saint Lucia,028,Antigua and Barbuda,000,World,Import,Sea,"081340 | Fruit, edible; fruit",81340,kg,6.807204,6.807204,16.850781,0.0,16.850781,"Antigua and Barbuda | 081340 | Fruit, edible; ...",08,"08 | Fruit and nuts, edib","Antigua and Barbuda | 08 | Fruit and nuts, edib",0813,"0813 | Fruit, dried, other","Antigua and Barbuda | 0813 | Fruit, dried, other",8.0,813.0,81340.0,Dried fruit (excl. of 08.01-08.06 & 0813.10-08...,Agriculture,113.0,"Growing of fruit, nuts, beverage and spice crops",5799.0,"Fruit, dried, n.e.s., & mixtures, n.e.s., of n...",4.0,"V_F - Vegetables, fruit, nuts",135.0,"Other fruit, dried",2.9373,0.3405,both,0.403970,5.737691,6.807204,[Container],['Container'],392.0,243.577432,1.397572,3 - Reporter and HS4 code,4.79,6,0.77,16.85,16.08,1. Primary Int. Inv. with Vess Type,0.007305,0.007292,0.007293,0.007293,0.007299,0.007299,0.007641,0.013543,0.008429,0.007825,0.007846,0.008043,0.007482,0.013263,0.008872,0.008126,0.007940,0.008299,0.007343,0.014423,0.011001,0.009621,0.008901,0.008650,0.007206,0.015335,0.012619,0.011825,0.009530,0.009271,0.007071,0.015540,0.014847,0.014974,0.010332,0.009739,0.006934,0.01522,0.014586,0.015981,0.011821,0.009917,0.006805,0.013577,0.014151,0.014828,0.011116,0.010035,0.006673,0.012471,0.012678,0.015814,0.013064,0.010101


## 2.4 Create Results Tables

### 2.4.1 Total Costs per Scenario

In [7]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_im["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * trade_im["direct_usd_{0}".format(year)].sum() / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["direct_usd_{0}".format(year)].sum() / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,4229349.57,4147103.67,4074986.50,4004270.51,3934871.08,3864180.08,3797647.44,3729879.52
1,Levy,7259984.32,7085682.27,7731741.52,8297771.98,8403523.87,8098756.26,7307387.90,6737504.75
2,'as-is',4599261.00,4830701.83,5909010.04,6794601.38,7970322.75,7818980.12,7567517.19,6870485.70
3,Brazil,4293935.41,4465328.81,5258817.72,6390861.18,8129671.50,8492384.28,8076020.51,8358715.86
4,LISCR,4307856.22,4374777.89,4894261.14,5189540.80,5607881.40,6426384.73,6167605.28,6915084.25
5,Direct,4428599.64,4557568.84,4736379.83,5054309.30,5293154.95,5382711.54,5440548.05,5488941.85



Total FEM costs by scenario (% ciffob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,11.71,11.49,11.29,11.09,10.90,10.70,10.52,10.33
1,Levy,20.11,19.63,21.41,22.98,23.28,22.43,20.24,18.66
2,'as-is',12.74,13.38,16.37,18.82,22.08,21.66,20.96,19.03
3,Brazil,11.89,12.37,14.57,17.70,22.52,23.52,22.37,23.15
4,LISCR,11.93,12.12,13.56,14.37,15.53,17.80,17.08,19.15
5,Direct,12.27,12.62,13.12,14.00,14.66,14.91,15.07,15.20



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.97,0.95,0.93,0.92,0.90,0.88,0.87,0.85
1,Levy,1.66,1.62,1.77,1.90,1.92,1.85,1.67,1.54
2,'as-is',1.05,1.10,1.35,1.55,1.82,1.79,1.73,1.57
3,Brazil,0.98,1.02,1.20,1.46,1.86,1.94,1.85,1.91
4,LISCR,0.98,1.00,1.12,1.19,1.28,1.47,1.41,1.58
5,Direct,1.01,1.04,1.08,1.16,1.21,1.23,1.24,1.25



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.05,1.03,1.02,1.00,0.98,0.96,0.95,0.93
1,Levy,1.81,1.76,1.93,2.07,2.09,2.02,1.82,1.68
2,'as-is',1.15,1.20,1.47,1.69,1.99,1.95,1.88,1.71
3,Brazil,1.07,1.11,1.31,1.59,2.02,2.12,2.01,2.08
4,LISCR,1.07,1.09,1.22,1.29,1.40,1.60,1.54,1.72
5,Direct,1.10,1.14,1.18,1.26,1.32,1.34,1.36,1.37



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.280,0.274,0.270,0.265,0.260,0.256,0.251,0.247
1,Levy,0.480,0.469,0.512,0.549,0.556,0.536,0.484,0.446
2,'as-is',0.304,0.320,0.391,0.450,0.527,0.517,0.501,0.455
3,Brazil,0.284,0.296,0.348,0.423,0.538,0.562,0.534,0.553
4,LISCR,0.285,0.290,0.324,0.343,0.371,0.425,0.408,0.458
5,Direct,0.293,0.302,0.313,0.335,0.350,0.356,0.360,0.363


### 2.4.2 Costs per Scenario relative to BAU

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im["bau_usd_{0}".format(year)].sum()), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_ciffob.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)

print("\nTotal FEM costs by scenario (% cif)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 2)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)

if res_mlc == True:
    print("\nTotal FEM costs by scenario (% fob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2),
            round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.TandT_fobvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
    display(df)
    
print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["direct_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,3030634.75,2938578.59,3656755.03,4293501.48,4468652.78,4234576.18,3509740.46,3007625.24
1,'as-is',369911.43,683598.16,1834023.54,2790330.88,4035451.66,3954800.04,3769869.74,3140606.19
2,Brazil,64585.83,318225.14,1183831.22,2386590.68,4194800.42,4628204.20,4278373.06,4628836.34
3,LISCR,78506.65,227674.22,819274.64,1185270.29,1673010.32,2562204.65,2369957.83,3185204.73
4,Direct,199250.07,410465.17,661393.34,1050038.79,1358283.87,1518531.46,1642900.61,1759062.33



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,71.66,70.86,89.74,107.22,113.57,109.59,92.42,80.64
1,'as-is',4.71,9.90,16.23,26.22,34.52,39.30,43.26,47.16
2,Brazil,8.75,16.48,45.01,69.68,102.56,102.35,99.27,84.20
3,LISCR,1.53,7.67,29.05,59.60,106.61,119.77,112.66,124.10
4,Direct,1.86,5.49,20.10,29.60,42.52,66.31,62.41,85.40



Total FEM costs by scenario (% ciffob)...



...

,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.69,0.67,0.84,0.98,1.02,0.97,0.80,0.69
1,'as-is',0.05,0.09,0.15,0.24,0.31,0.35,0.38,0.40
2,Brazil,0.08,0.16,0.42,0.64,0.92,0.90,0.86,0.72
3,LISCR,0.01,0.07,0.27,0.55,0.96,1.06,0.98,1.06
4,Direct,0.02,0.05,0.19,0.27,0.38,0.59,0.54,0.73



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.75,0.73,0.91,1.07,1.11,1.05,0.87,0.75
1,'as-is',0.05,0.10,0.16,0.26,0.34,0.38,0.41,0.44
2,Brazil,0.09,0.17,0.46,0.70,1.01,0.99,0.94,0.78
3,LISCR,0.02,0.08,0.29,0.59,1.04,1.15,1.07,1.15
4,Direct,0.02,0.06,0.20,0.30,0.42,0.64,0.59,0.79



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.201,0.194,0.242,0.284,0.296,0.280,0.232,0.199
1,'as-is',0.013,0.027,0.044,0.069,0.090,0.100,0.109,0.116
2,Brazil,0.024,0.045,0.121,0.185,0.267,0.262,0.249,0.208
3,LISCR,0.004,0.021,0.078,0.158,0.278,0.306,0.283,0.306
4,Direct,0.005,0.015,0.054,0.078,0.111,0.170,0.157,0.211


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [9]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str", "HS4":"str"}
if res_mlc == False:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
else:
    trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_mlc_{1}.csv".format(iso_country, version), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Saint Lucia': 

Number of trade records present in dataset: 3053
Number of partner countries present in dataset: 59
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$50.7m
Total weight within the dataset: 176.6 million kilograms

Distance to market: 83.3 nm (min) to 13463.9 nm (max), 1424.5 nm (observed mean)
Average distance to market (volume-weighted): 1186.3 nm
Average distance to market (value-weighted): 1355.9 nm


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue
0,0,662,Saint Lucia,028,Antigua and Barbuda,000,World,Export,Sea,"091012 | Spices; ginger, crus",91012,kg,8.27190,8.271900,0.0,34.034273,34.034273,"Antigua and Barbuda | 091012 | Spices; ginger,...",09,"09 | Coffee, tea, mate an","Antigua and Barbuda | 09 | Coffee, tea, mate an",0910,"0910 | Ginger, saffron, tum","Antigua and Barbuda | 0910 | Ginger, saffron, tum",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.1973,0.4551,both,0.243046,15.488998,8.271900,[Container],['Container'],392.0,243.577432,3.772770,"2 - Reporter, Partner and HS2 code",3.75,1,1.28,35.31,34.03
1,1,662,Saint Lucia,028,Antigua and Barbuda,000,World,Export,Sea,220840 | Rum and other spirit,220840,l,7.05442,10.924818,0.0,274.559933,274.559933,Antigua and Barbuda | 220840 | Rum and other s...,22,"22 | Beverages, spirits a","Antigua and Barbuda | 22 | Beverages, spirits a",2208,"2208 | Ethyl alcohol, unden","Antigua and Barbuda | 2208 | Ethyl alcohol, unden",22.0,2208.0,220840.0,Rum & tafia,Food & Beverages,1551.0,"Distilling, rectifying and blending of spirits...",11244.0,Rum & tafia,26.0,B_T - Beverages and tobacco products,2413.0,Undenatured ethyl alcohol of an alcoholic stre...,6.3251,0.1581,both,0.039790,43.407925,10.924818,[Container],['Container'],392.0,243.577432,10.573191,"1 - Reporter, Partner and HS4 code",2.78,1,7.63,282.19,274.56


## 3.2 Derive Internationally Departing Voyages

In [10]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 339
Total size of the fleet: 136 vessels
6 vessel types contained in the dataset: ['Vehicle' 'Oil tanker' 'General cargo' 'Container' 'Chemical tanker'
 'Bulk carrier']
Total transport work capacity: 16021.0 million deadweight-tonne nautical miles
Total cargo transported: 5884997.2 tonnes
Total transport work: 7871.7 million tonne nautical miles
Average build year of the fleet: 2007
Total Energy Demands: 1152.3 EJ
Total CO2 produced: 88788.7 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,vessel_type,gross_tonnage,capacity_unit,deadweight,teu,gas_cap,passengers,cars,size_id,size_range,year_of_built,flag,engine_type,me_fuel,class_med_aer,class_util,bau35_usd_dwtnm_26,bau35_usd_dwtnm_29,bau35_usd_dwtnm_32,bau35_usd_dwtnm_35,bau35_usd_dwtnm_38,bau35_usd_dwtnm_41,bau35_usd_dwtnm_44,bau35_usd_dwtnm_47,bau35_usd_dwtnm_50,levy35_usd_dwtnm_26,levy35_usd_dwtnm_29,levy35_usd_dwtnm_32,levy35_usd_dwtnm_35,levy35_usd_dwtnm_38,levy35_usd_dwtnm_41,levy35_usd_dwtnm_44,levy35_usd_dwtnm_47,levy35_usd_dwtnm_50,asis35_usd_dwtnm_26,asis35_usd_dwtnm_29,asis35_usd_dwtnm_32,asis35_usd_dwtnm_35,asis35_usd_dwtnm_38,asis35_usd_dwtnm_41,asis35_usd_dwtnm_44,asis35_usd_dwtnm_47,asis35_usd_dwtnm_50,brazil35_usd_dwtnm_26,brazil35_usd_dwtnm_29,brazil35_usd_dwtnm_32,brazil35_usd_dwtnm_35,brazil35_usd_dwtnm_38,brazil35_usd_dwtnm_41,brazil35_usd_dwtnm_44,brazil35_usd_dwtnm_47,brazil35_usd_dwtnm_50,liscr35_usd_dwtnm_26,liscr35_usd_dwtnm_29,liscr35_usd_dwtnm_32,liscr35_usd_dwtnm_35,liscr35_usd_dwtnm_38,liscr35_usd_dwtnm_41,liscr35_usd_dwtnm_44,liscr35_usd_dwtnm_47,liscr35_usd_dwtnm_50,direct35_usd_dwtnm_26,direct35_usd_dwtnm_29,direct35_usd_dwtnm_32,direct35_usd_dwtnm_35,direct35_usd_dwtnm_38,direct35_usd_dwtnm_41,direct35_usd_dwtnm_44,direct35_usd_dwtnm_47,direct35_usd_dwtnm_50,dwtnm,bau35_usd_26,bau35_usd_29,bau35_usd_32,bau35_usd_35,bau35_usd_38,bau35_usd_41,bau35_usd_44,bau35_usd_47,bau35_usd_50,levy35_usd_26,levy35_usd_29,levy35_usd_32,levy35_usd_35,levy35_usd_38,levy35_usd_41,levy35_usd_44,levy35_usd_47,levy35_usd_50,asis35_usd_26,asis35_usd_29,asis35_usd_32,asis35_usd_35,asis35_usd_38,asis35_usd_41,asis35_usd_44,asis35_usd_47,asis35_usd_50,brazil35_usd_26,brazil35_usd_29,brazil35_usd_32,brazil35_usd_35,brazil35_usd_38,brazil35_usd_41,brazil35_usd_44,brazil35_usd_47,brazil35_usd_50,liscr35_usd_26,liscr35_usd_29,liscr35_usd_32,liscr35_usd_35,liscr35_usd_38,liscr35_usd_41,liscr35_usd_44,liscr35_usd_47,liscr35_usd_50,direct35_usd_26,direct35_usd_29,direct35_usd_32,direct35_usd_35,direct35_usd_38,direct35_usd_41,direct35_usd_44,direct35_usd_47,direct35_usd_50,tonnes,tnm
0,79198,9078830,353910000,LCCAS,662,LCA,LC,Saint Lucia,Castries,BBBGI,052,BRB,BB,Barbados,Bridgetown,2018-12-08 03:00:00,2018-12-08 13:00:00,10.0,30.0,40.0,125.817086,9.244003,7.033492,16.277495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.371609,0.282746,28.785825,21.902293,50.688118,019,Americas,419,Latin America and the Caribbean,upp_mid_income,0,1,019,Americas,419,Latin America and the Caribbean,high_income,0,1,0.654355,Vehicle,49012.0,gt,15553.0,NaN,NaN,NaN,5008.0,2,30000-49999,1994.0,Panama,SSD,1,21.8,0.292985,0.010079,0.010502,0.010302,0.010127,0.009955,0.009786,0.009614,0.009452,0.009287,0.010033,0.01817,0.016375,0.018434,0.020218,0.018328,0.017672,0.015414,0.014738,0.01004,0.011031,0.011371,0.01404,0.016494,0.016992,0.014587,0.017643,0.015442,0.01004,0.010325,0.010829,0.012868,0.015565,0.018792,0.016586,0.019151,0.01784,0.01006,0.010413,0.010741,0.012181,0.012203,0.013616,0.011002,0.014594,0.012311,0.01006,0.01089,0.011148,0.011518,0.012217,0.012709,0.012855,0.01293,0.012958,1.956833e+06,19723.807423,20550.257584,20159.323515,19816.455090,19479.938862,19149.774720,18812.992373,18495.719340,18172.201758,19632.996381,35554.956852,32042.612177,36071.293781,39564.107072,35865.345762,34581

## 3.3 Evaluate NZF Costs

In [11]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
vess_types_not_found_in_inventories, vess_types_not_found_in_global_voys, hs_codes_not_mapped_to_HS, cost_int_match_desc, cost_int_match_desc_bool = [], [], [], [], False

for year in years:

    print("Processing game year 20{0}".format(year))

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, asis_usd, brazil_usd, liscr_usd, direct_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # initiate selected voyages table as empty dataframe to begin
        int_dep_voys_vess = pd.DataFrame()

        # 3. If no facilitating vessel types were picked up for the HS code, take from full primary int. inv. voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.HS4_desc)
            int_dep_voys_vess = int_dep_voys
            cost_int_match_desc.append("3. Primary Int. Inv. without Vess Type") if cost_int_match_desc_bool == False else None

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]    
            
            if int_dep_voys_vess.shape[0] > 0:
                cost_int_match_desc.append("1. Primary Int. Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                
            else:
                # 2. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
                
                if int_dep_voys_vess.shape[0] > 0:
                    cost_int_match_desc.append("2. Global Inv. with Vess Type") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]

                else:
                    # 4. Use full global voyage dataset as a back-up
                    int_dep_voys_vess = voys_fem
                    cost_int_match_desc.append("4. Global Inv. without Vess Type.") if cost_int_match_desc_bool == False else None
                    [vess_types_not_found_in_global_voys.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
    
        # Add cost intensities associated with the subset of voyages by vessel type, assuming 50% of NZF costs are borne by the exporter.
        bau_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau35_usd_dwtnm_"+year))
        levy_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy35_usd_dwtnm_"+year))
        asis_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis35_usd_dwtnm_"+year))
        brazil_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil35_usd_dwtnm_"+year))
        liscr_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr35_usd_dwtnm_"+year))
        direct_usd.append(0.5 * trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "direct35_usd_dwtnm_"+year))

    if cost_int_match_desc_bool == False:
        trade_ex["cost_int_match_desc"] = cost_int_match_desc
    cost_int_match_desc_bool = True

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd
    trade_ex["direct_usd_"+year] = direct_usd
    
print("Inventory voyages table used for the following vessel types: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))
print("Global voyages table used for the following vessel types found: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_global_voys).unique()))
print("Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.vol_kg_vtw.sum() / 1e6, 1)))

print("\nDistance to market: {0} nm (min) to {1} nm (max), {2} nm (observed mean)".format(
    round(trade_ex.sea_dist_nm.min(), 1),
    round(trade_ex.sea_dist_nm.max(), 1),
    round(trade_ex.sea_dist_nm.mean(), 1)
))
print("Average distance to market (volume-weighted): {0} nm".format(
    round((trade_ex.vol_kg_vtw / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.vol_kg_vtw.sum() / 1000), 1)
))
print("Average distance to market (value-weighted): {0} nm".format(
    round((trade_ex.fobvalue / 1000 * trade_ex.sea_dist_nm).sum() / (trade_ex.fobvalue.sum() / 1000), 1)
))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Direct: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.direct_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.direct_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_agg = trade_ex.groupby(["cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg["vol_kg_vtw_pct"] = round(100 * trade_ex_agg.vol_kg_vtw / trade_ex_agg.vol_kg_vtw.sum(), 1)
trade_ex_agg["fobvalue_pct"] = round(100 * trade_ex_agg.fobvalue / trade_ex_agg.fobvalue.sum(), 1)
print("Most prominent 'cost intensity match' types...\n\n{0}\n".format(
    trade_ex_agg[["cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5)
))
trade_ex_agg_2 = trade_ex.groupby(["fac_vessel_types_str", "cost_int_match_desc"]).agg({"vol_kg_vtw":"sum", "fobvalue":"sum"}).reset_index().sort_values(by="vol_kg_vtw", ascending=False).reset_index(drop=True)
trade_ex_agg_2["vol_kg_vtw_pct"] = round(100 * trade_ex_agg_2.vol_kg_vtw / trade_ex_agg_2.vol_kg_vtw.sum(), 1)
trade_ex_agg_2["fobvalue_pct"] = round(100 * trade_ex_agg_2.fobvalue / trade_ex_agg_2.fobvalue.sum(), 1)
print("Most prominent 'facilitating vessel types' and their 'cost intensity match' types...")
display(trade_ex_agg_2[["fac_vessel_types_str", "cost_int_match_desc", "vol_kg_vtw_pct", "fobvalue_pct"]].head(5))

trade_ex_na = trade_ex[(trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
trade_ex = trade_ex[(~trade_ex[["vol_kg_vtw", "primaryValue", "HS2", "HS6", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n".format(trade_ex_na.primaryValue.sum(), round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Processing game year 2026
Processing game year 2029
Processing game year 2032
Processing game year 2035
Processing game year 2038
Processing game year 2041
Processing game year 2044
Processing game year 2047
Processing game year 2050
Inventory voyages table used for the following vessel types: 

['Liquefied gas tanker']

Global voyages table used for the following vessel types found: 

['Ro-Ro']

Global voyages table used for the following commodities as no 'facilitating vessel types' were identified: 

['2106 | Food preparations no' '2202 | Waters, including mi'
 '8528 | Monitors and project' '0714 | Manioc, arrowroot, s'
 '0910 | Ginger, saffron, tum' '3822 | Reagents; diagnostic'
 '7326 | Iron or steel; artic' '0805 | Citrus fruit; fresh '
 '3926 | Articles of plastics' '4911 | Printed matter, n.e.'
 '4421 | Wooden articles n.e.' '8215 | Cutlery; spoons, for'
 '8525 | Transmission apparat' '3824 | Prepared binders for'
 '1806 | Chocolate and other ' '4823 | Paper, paperboard, c'
 '2

,fac_vessel_types_str,cost_int_match_desc,vol_kg_vtw_pct,fobvalue_pct
0,"['Container', 'General cargo']",1. Primary Int. Inv. with Vess Type,72.0,4.0
1,['Container'],1. Primary Int. Inv. with Vess Type,21.4,73.5
2,"['Container', 'Bulk carrier']",1. Primary Int. Inv. with Vess Type,2.0,2.9
3,['Oil tanker'],1. Primary Int. Inv. with Vess Type,1.8,4.1
4,['all'],3. Primary Int. Inv. without Vess Type,1.0,6.2


Rows with NA: $0.0 (0.0%)



,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,HS6_desc,HS6,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,HS6_bilat_desc,HS2,HS2_desc,HS2_bilat_desc,HS4,HS4_desc,HS4_bilat_desc,hs2,hs4,hs6,hs6_description,eora_desc,isic,isic_desc,sitc,sitc_desc,gtap,gtap_desc,cpc,cpc_desc,usd_per_kg,kg_per_usd_cepii,_trade_ex_vtw_hs4_m,kg_per_usd_netWgt,vol_kg_vtw,vol_kg_vtw_comp,fac_vessel_types,fac_vessel_types_str,sea_dist_km,sea_dist_nm,tnm,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50
0,0,662,Saint Lucia,028,Antigua and Barbuda,000,World,Export,Sea,"091012 | Spices; ginger, crus",91012,kg,8.27190,8.271900,0.0,34.034273,34.034273,"Antigua and Barbuda | 091012 | Spices; ginger,...",09,"09 | Coffee, tea, mate an","Antigua and Barbuda | 09 | Coffee, tea, mate an",0910,"0910 | Ginger, saffron, tum","Antigua and Barbuda | 0910 | Ginger, saffron, tum",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.1973,0.4551,both,0.243046,15.488998,8.271900,[Container],['Container'],392.0,243.577432,3.772770,"2 - Reporter, Partner and HS2 code",3.75,1,1.28,35.31,34.03,1. Primary Int. Inv. with Vess Type,0.019720,0.019685,0.019689,0.019689,0.019704,0.019704,0.020628,0.036559,0.022753,0.021124,0.021181,0.021712,0.020198,0.035804,0.023949,0.021937,0.021434,0.022404,0.019821,0.038934,0.029698,0.025973,0.024030,0.023352,0.019452,0.041397,0.034065,0.031921,0.025727,0.025027,0.019089,0.041950,0.040079,0.040424,0.027891,0.026290,0.018719,0.041087,0.039374,0.043140,0.031911,0.026772,0.018370,0.036651,0.038200,0.040030,0.030007,0.027089,0.018015,0.033667,0.034223,0.042689,0.035267,0.027268
1,1,662,Saint Lucia,028,Antigua and Barbuda,000,World,Export,Sea,220840 | Rum and other spirit,220840,l,7.05442,10.924818,0.0,274.559933,274.559933,Antigua and Barbuda | 220840 | Rum and other s...,22,"22 | Beverages, spirits a","Antigua and Barbuda | 22 | Beverages, spirits a",2208,"2208 | Ethyl alcohol, unden","Antigua and Barbuda | 2208 | Ethyl alcohol, unden",22.0,2208.0,220840.0,Rum & tafia,Food & Beverages,1551.0,"Distilling, rectifying and blending of spirits...",11244.0,Rum & tafia,26.0,B_T - Beverages and tobacco products,2413.0,Undenatured ethyl alcohol of an alcoholic stre...,6.3251,0.1581,both,0.039790,43.407925,10.924818,[Container],['Container'],392.0,243.577432,10.573191,"1 - Reporter, Partner and HS4 code",2.78,1,7.63,282.19,274.56,1. Primary Int. Inv. with Vess Type,0.055264,0.055168,0.055177,0.055177,0.055220,0.055220,0.057809,0.102456,0.063765,0.059200,0.059361,0.060848,0.056605,0.100340,0.067117,0.061478,0.060067,0.062788,0.055550,0.109113,0.083228,0.072789,0.067343,0.065444,0.054513,0.116014,0.095466,0.089460,0.072100,0.070139,0.053497,0.117564,0.112322,0.113288,0.078164,0.073679,0.052460,0.115147,0.110346,0.120901,0.089429,0.075029,0.051482,0.102714,0.107056,0.112183,0.084096,0.075916,0.050486,0.094352,0.095911,0.119637,0.098837,0.076419


## 3.4 Create Results Tables

### 3.4.1 Total Costs per Scenario

In [12]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum(), 2),
        np.round(trade_ex["direct_usd_{0}".format(year)].sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.TandT_cifvalue.sum(), 2)
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)

    
print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["direct_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1288810.58,1264712.06,1243552.56,1222788.81,1202409.87,1181625.64,1162047.90,1142084.72
1,Levy,2187429.22,2119652.83,2319998.91,2478908.38,2493759.05,2444368.98,2203771.17,2039798.62
2,'as-is',1397734.73,1461711.87,1787937.38,2036400.02,2396769.40,2341260.47,2273712.33,2065675.72
3,Brazil,1311387.95,1361261.93,1592918.48,1922812.28,2428247.75,2523739.13,2398117.70,2489197.04
4,LISCR,1311053.65,1333138.73,1487841.04,1570042.88,1691332.80,1932491.23,1846317.87,2066773.77
5,Direct,1347859.99,1386168.79,1438697.37,1532045.23,1601878.51,1628222.39,1645430.96,1655448.51



Total FEM costs by scenario (% ciffob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,25.55,25.07,24.66,24.24,23.84,23.43,23.04,22.64
1,Levy,43.37,42.03,46.00,49.15,49.44,48.46,43.69,40.44
2,'as-is',27.71,28.98,35.45,40.37,47.52,46.42,45.08,40.96
3,Brazil,26.00,26.99,31.58,38.12,48.14,50.04,47.55,49.35
4,LISCR,25.99,26.43,29.50,31.13,33.53,38.31,36.61,40.98
5,Direct,26.72,27.48,28.52,30.38,31.76,32.28,32.62,32.82



Total FEM costs by scenario (% cif)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.31,2.27,2.23,2.20,2.16,2.12,2.09,2.05
1,Levy,3.93,3.81,4.16,4.45,4.48,4.39,3.96,3.66
2,'as-is',2.51,2.62,3.21,3.66,4.30,4.20,4.08,3.71
3,Brazil,2.35,2.44,2.86,3.45,4.36,4.53,4.31,4.47
4,LISCR,2.35,2.39,2.67,2.82,3.04,3.47,3.31,3.71
5,Direct,2.42,2.49,2.58,2.75,2.88,2.92,2.95,2.97



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.54,2.50,2.45,2.41,2.37,2.33,2.29,2.25
1,Levy,4.32,4.18,4.58,4.89,4.92,4.82,4.35,4.03
2,'as-is',2.76,2.89,3.53,4.02,4.73,4.62,4.49,4.08
3,Brazil,2.59,2.69,3.14,3.80,4.79,4.98,4.73,4.91
4,LISCR,2.59,2.63,2.94,3.10,3.34,3.81,3.64,4.08
5,Direct,2.66,2.74,2.84,3.02,3.16,3.21,3.25,3.27



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.085,0.084,0.082,0.081,0.080,0.078,0.077,0.076
1,Levy,0.145,0.140,0.154,0.164,0.165,0.162,0.146,0.135
2,'as-is',0.093,0.097,0.118,0.135,0.159,0.155,0.150,0.137
3,Brazil,0.087,0.090,0.105,0.127,0.161,0.167,0.159,0.165
4,LISCR,0.087,0.088,0.098,0.104,0.112,0.128,0.122,0.137
5,Direct,0.089,0.092,0.095,0.101,0.106,0.108,0.109,0.110


### 3.4.2 Costs per Scenario relative to BAU

In [13]:
print("\nTotal FEM costs by scenario (US$)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        np.round((trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_usd_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal change in FEM costs by scenario (% bau)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        round((100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
        round((100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex["bau_usd_{0}".format(year)].sum()), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_bau_{1}.csv".format(iso_country, version), index=False)
display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% ciffob)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_ciffob.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_ciffob_{1}.csv".format(iso_country, version), index=False)
    display(df)


if res_mlc == True:
    print("\nTotal FEM costs by scenario (% cif)...\n")
    d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
    for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
        d_temp = {"20{0}".format(year): [
            round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
            round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.TandT_cifvalue.sum(), 2),
        ]}
        d = d | d_temp
    df = pd.DataFrame(data=d)
    df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_cif_{1}.csv".format(iso_country, version), index=False)
    display(df)


print("\nTotal FEM costs by scenario (% fob)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 2),
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_fob_{1}.csv".format(iso_country, version), index=False)
display(df)


print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "'as-is'", "Brazil", "LISCR", "Direct"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["direct_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/results/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,898618.64,854940.77,1076446.35,1256119.57,1291349.17,1262743.34,1041723.27,897713.90
1,'as-is',108924.15,196999.81,544384.82,813611.21,1194359.53,1159634.83,1111664.43,923591.00
2,Brazil,22577.37,96549.87,349365.93,700023.47,1225837.87,1342113.49,1236069.80,1347112.33
3,LISCR,22243.07,68426.66,244288.48,347254.07,488922.93,750865.59,684269.96,924689.06
4,Direct,59049.41,121456.72,195144.81,309256.42,399468.63,446596.75,483383.06,513363.79



Total change in FEM costs by scenario (% bau)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,69.72,67.60,86.56,102.73,107.40,106.86,89.65,78.60
1,'as-is',8.45,15.58,43.78,66.54,99.33,98.14,95.66,80.87
2,Brazil,1.75,7.63,28.09,57.25,101.95,113.58,106.37,117.95
3,LISCR,1.73,5.41,19.64,28.40,40.66,63.55,58.88,80.97
4,Direct,4.58,9.60,15.69,25.29,33.22,37.80,41.60,44.95



Total FEM costs by scenario (% ciffob)...



...

,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.61,1.53,1.93,2.25,2.32,2.27,1.87,1.61
1,'as-is',0.20,0.35,0.98,1.46,2.14,2.08,2.00,1.66
2,Brazil,0.04,0.17,0.63,1.26,2.20,2.41,2.22,2.42
3,LISCR,0.04,0.12,0.44,0.62,0.88,1.35,1.23,1.66
4,Direct,0.11,0.22,0.35,0.56,0.72,0.80,0.87,0.92



Total FEM costs by scenario (% fob)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.77,1.69,2.12,2.48,2.55,2.49,2.06,1.77
1,'as-is',0.22,0.39,1.07,1.61,2.36,2.29,2.19,1.82
2,Brazil,0.04,0.19,0.69,1.38,2.42,2.65,2.44,2.66
3,LISCR,0.04,0.14,0.48,0.69,0.97,1.48,1.35,1.83
4,Direct,0.12,0.24,0.39,0.61,0.79,0.88,0.95,1.01



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.059,0.057,0.071,0.083,0.085,0.084,0.069,0.059
1,'as-is',0.007,0.013,0.036,0.054,0.079,0.077,0.074,0.061
2,Brazil,0.001,0.006,0.023,0.046,0.081,0.089,0.082,0.089
3,LISCR,0.001,0.005,0.016,0.023,0.032,0.050,0.045,0.061
4,Direct,0.004,0.008,0.013,0.020,0.026,0.030,0.032,0.034
